# K-Suiter: four-patient training, one-patient Gen1 test

This notebook implements a patient-disjoint experiment for the only Siena patients with at least four usable seizures: `PN00`, `PN06`, `PN10`, `PN12`, and `PN14`.

Enter exactly four of those patients below. The remaining patient is inferred automatically and is kept completely untouched during sensor selection and model fitting. K-Suiter pools **all seizures and matched controls** from the four training patients. Candidate sensor sets are compared by leave-one-training-patient-out validation, after which Gen1 is fitted once on all four training patients and evaluated on every seizure from the fifth patient.

There is no random seizure split in this notebook. The split boundary is the patient ID.

> **Research only:** this experiment has not been clinically validated and must not directly control patient care.

## 1. Inputs

Patient values may be integers (`0`) or Siena strings (`"PN00"`). `K` is loaded from the versioned K-Finder result.

In [1]:
TRAIN_PATIENTS = [0, 6, 10, 12]
FORCE_REBUILD_CHANNEL_FEATURES = False
FORCE_REBUILD_GEN1_FEATURES = False

## 2. Setup and validate the patient-disjoint split

In [2]:
from dataclasses import replace
from pathlib import Path
from typing import Iterable
import hashlib
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, brier_score_loss

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "k_suiter.py").exists():
    candidates = list(NOTEBOOK_DIR.rglob("k_suiter.py"))
    if len(candidates) != 1:
        raise FileNotFoundError(
            "Run this notebook from the scripts directory or repository root."
        )
    NOTEBOOK_DIR = candidates[0].parent
sys.path.insert(0, str(NOTEBOOK_DIR))

import k_suiter as ks
import personalized_channels_workflow as pc
import rolling_seizure_forecasting as rsf

ELIGIBLE_PATIENTS = ("PN00", "PN06", "PN10", "PN12", "PN14")

def normalize_patient_id(value: int | str) -> str:
    text = str(value).strip().upper()
    if text.startswith("PN"):
        text = text[2:]
    if not text.isdigit():
        raise ValueError(f"Invalid Siena patient ID: {value!r}")
    return f"PN{int(text):02d}"

training_patients = tuple(normalize_patient_id(value) for value in TRAIN_PATIENTS)
if len(training_patients) != 4 or len(set(training_patients)) != 4:
    raise ValueError("TRAIN_PATIENTS must contain exactly four distinct patients.")
invalid = sorted(set(training_patients) - set(ELIGIBLE_PATIENTS))
if invalid:
    raise ValueError(
        f"Only {list(ELIGIBLE_PATIENTS)} may be used; received invalid IDs {invalid}."
    )
test_patient = next(iter(set(ELIGIBLE_PATIENTS) - set(training_patients)))

paths = pc.personalized_paths(NOTEBOOK_DIR)
k_finder_result = ks.load_k_finder_result(paths["project"])
K = k_finder_result.k
selector_config = pc.PersonalizedConfig(
    k=K,
    patient_ids=training_patients,
    swap_refinement=False,
    force_rebuild_features=FORCE_REBUILD_CHANNEL_FEATURES,
)
selector_config.validate()
forecast_config = rsf.ForecastConfig(
    test_fraction=0.20,
    random_seed=42,
    interictal_controls_per_seizure=4,
    max_iter=60,
)

print(f"Training patients: {training_patients}")
print(f"Untouched test patient: {test_patient}")
print(f"K={K} (from {k_finder_result.source_path.name})")

Training patients: ('PN00', 'PN06', 'PN10', 'PN12')
Untouched test patient: PN14
K=12 (from sensor_count_selected.json)


## 3. Load all five eligible patients

The held-out patient's channel names are inspected only to ensure the selected montage can physically be evaluated there. Its signal-derived features and outcomes never enter sensor scoring.

In [3]:
manifest = pc.load_manifest(paths, forecast_config)
cohort_manifest = manifest.loc[
    manifest["patient_id"].astype(str).isin(ELIGIBLE_PATIENTS)
].copy()

event_counts = (
    cohort_manifest.loc[cohort_manifest["episode_type"].eq("preictal")]
    .groupby("patient_id")["source_event_id"]
    .nunique()
    .reindex(ELIGIBLE_PATIENTS, fill_value=0)
)
too_small = event_counts[event_counts < 4]
if not too_small.empty:
    raise ValueError(
        "Every allowed patient must have at least four usable seizures; found "
        + str(too_small.to_dict())
    )

patient_data = {}
for patient_id in training_patients:
    patient_manifest = cohort_manifest.loc[
        cohort_manifest["patient_id"].astype(str).eq(patient_id)
    ].copy()
    patient_data[patient_id] = pc.build_patient_feature_data(
        patient_manifest,
        paths["feature_cache"],
        forecast_config,
        force=FORCE_REBUILD_CHANNEL_FEATURES,
    )

test_manifest = cohort_manifest.loc[
    cohort_manifest["patient_id"].astype(str).eq(test_patient)
].copy()
test_channel_names = pc.common_patient_channels(test_manifest)
common_channels = set.intersection(
    *(set(patient_data[patient_id].channel_names) for patient_id in training_patients),
    set(test_channel_names),
)
common_channels = sorted(common_channels, key=pc._natural_recording_key)
if K > len(common_channels):
    raise ValueError(
        f"K={K}, but only {len(common_channels)} channels exist in all five patients."
    )

def compact_map(data: pc.PatientFeatureData) -> dict[str, list[str]]:
    try:
        result = pc.compact_channel_column_map(data.channel_feature_columns)
    except (AssertionError, ValueError):
        result = data.channel_feature_columns
    return {channel: list(result[channel]) for channel in common_channels}

patient_channel_maps = {
    patient_id: compact_map(patient_data[patient_id])
    for patient_id in training_patients
}
reference_map = patient_channel_maps[training_patients[0]]
for patient_id, channel_map in patient_channel_maps.items():
    mismatched = [
        channel for channel in common_channels
        if channel_map[channel] != reference_map[channel]
    ]
    if mismatched:
        raise ValueError(
            f"Feature schema differs for {patient_id}: {mismatched[:5]}"
        )

training_frames = {
    patient_id: patient_data[patient_id].frame.copy()
    for patient_id in training_patients
}
split_overview = pd.DataFrame({
    "patient_id": ELIGIBLE_PATIENTS,
    "role": ["train" if p in training_patients else "test" for p in ELIGIBLE_PATIENTS],
    "usable_seizures": [int(event_counts[p]) for p in ELIGIBLE_PATIENTS],
    "channel_feature_rows_used_for_selection": [
        len(patient_data[p].frame) if p in patient_data else 0
        for p in ELIGIBLE_PATIENTS
    ],
})
display(split_overview)
print(f"Channels available in every patient: {len(common_channels)}")

PN00: loaded cached channel features.
PN06: loaded cached channel features.
PN10 [1/50] PN10_S01_preictal
PN10 [2/50] PN10_S02_preictal
PN10 [3/50] PN10_S03_preictal
PN10 [4/50] PN10_S04_preictal
PN10 [5/50] PN10_S05_preictal
PN10 [6/50] PN10_S06_preictal
PN10 [7/50] PN10_S07_preictal
PN10 [8/50] PN10_S08_preictal
PN10 [9/50] PN10_S09_preictal
PN10 [10/50] PN10_S10_preictal
PN10 [11/50] PN10_interictal_01
PN10 [12/50] PN10_interictal_02
PN10 [13/50] PN10_interictal_03
PN10 [14/50] PN10_interictal_04
PN10 [15/50] PN10_interictal_05
PN10 [16/50] PN10_interictal_06
PN10 [17/50] PN10_interictal_07
PN10 [18/50] PN10_interictal_08
PN10 [19/50] PN10_interictal_09
PN10 [20/50] PN10_interictal_10
PN10 [21/50] PN10_interictal_11
PN10 [22/50] PN10_interictal_12
PN10 [23/50] PN10_interictal_13
PN10 [24/50] PN10_interictal_14
PN10 [25/50] PN10_interictal_15
PN10 [26/50] PN10_interictal_16
PN10 [27/50] PN10_interictal_17
PN10 [28/50] PN10_interictal_18
PN10 [29/50] PN10_interictal_19
PN10 [30/50] PN

,patient_id,role,usable_seizures,channel_feature_rows_used_for_selection
0,PN00,train,5,1500
1,PN06,train,5,1500
2,PN10,train,10,3000
3,PN12,train,4,1200
4,PN14,test,4,0


Channels available in every patient: 29


## 4. Select K sensors from the four training patients

For each candidate montage, four validation fits are made: three training patients fit the selector and the fourth training patient validates it. Scores are averaged across those four patients. The fifth patient is never a validation fold.

In [4]:
def cohort_selection_score(channels: Iterable[str]) -> dict[str, float]:
    columns = pc._subset_columns(reference_map, channels)
    fold_rows = []
    all_truth = []
    all_probability = []
    for validation_patient in training_patients:
        fit = pd.concat(
            [frame for patient_id, frame in training_frames.items()
             if patient_id != validation_patient],
            ignore_index=True,
        )
        validation = training_frames[validation_patient]
        fit_risk, validation_risk = ks._selector_probabilities(
            fit, validation, columns, selector_config
        )
        threshold, _ = rsf.select_warning_threshold(
            fit, fit_risk, target_time_in_warning=0.25
        )
        utility, sensitivity, false_alarms, warning_time, auroc = ks._alarm_utility(
            validation, validation_risk, threshold
        )
        truth = validation["has_event_in_5m"].to_numpy(dtype=int)
        fold_rows.append((utility, sensitivity, false_alarms, warning_time, auroc))
        all_truth.append(truth)
        all_probability.append(validation_risk)
    means = np.mean(fold_rows, axis=0)
    truth = np.concatenate(all_truth)
    probability = np.concatenate(all_probability)
    return {
        "validation_alarm_utility": float(means[0]),
        "validation_sensitivity": float(means[1]),
        "validation_false_alarms_per_hour": float(means[2]),
        "validation_time_in_warning": float(means[3]),
        "validation_auroc": float(means[4]),
        "validation_auprc": float(average_precision_score(truth, probability)),
        "validation_brier": float(brier_score_loss(truth, probability)),
    }

selected_channels = []
remaining_channels = set(common_channels)
score_cache = {}
ranking_rows = []
for rank in range(1, K + 1):
    candidates = []
    for channel in sorted(remaining_channels, key=pc._natural_recording_key):
        subset = tuple(sorted([*selected_channels, channel], key=pc._natural_recording_key))
        if subset not in score_cache:
            score_cache[subset] = cohort_selection_score(subset)
        candidates.append((score_cache[subset]["validation_alarm_utility"], channel, subset))
    _, chosen, chosen_subset = max(candidates, key=lambda item: (item[0], item[1]))
    selected_channels.append(chosen)
    remaining_channels.remove(chosen)
    row = {
        "rank": rank,
        "channel": chosen,
        "channels": ", ".join(selected_channels),
        **score_cache[chosen_subset],
    }
    ranking_rows.append(row)
    print(f"Rank {rank}/{K}: added {chosen}")

ranking = pd.DataFrame(ranking_rows)
display(ranking.style.format({column: "{:.4f}" for column in ranking.columns if column.startswith("validation_")}))
print(f"Selected channels: {selected_channels}")

Rank 1/12: added T5
Rank 2/12: added PZ
Rank 3/12: added F8
Rank 4/12: added FP1
Rank 5/12: added F10
Rank 6/12: added CP5
Rank 7/12: added FC1
Rank 8/12: added F9
Rank 9/12: added CP1
Rank 10/12: added FC2
Rank 11/12: added P3
Rank 12/12: added O1


,rank,channel,channels,validation_alarm_utility,validation_sensitivity,validation_false_alarms_per_hour,validation_time_in_warning,validation_auroc,validation_auprc,validation_brier
0,1,T5,T5,0.5843,0.7000,7.9500,0.4660,0.4789,0.1945,0.2823
1,2,PZ,"T5, PZ",0.6211,0.7375,8.4375,0.4699,0.5365,0.2390,0.2916
2,3,F8,"T5, PZ, F8",0.6936,0.6250,5.8125,0.3260,0.5318,0.2291,0.2898
3,4,FP1,"T5, PZ, F8, FP1",0.7235,0.6875,6.8250,0.3098,0.5421,0.2364,0.2803
4,5,F10,"T5, PZ, F8, FP1, F10",0.9100,0.7625,6.4500,0.3106,0.5383,0.2771,0.2908
5,6,CP5,"T5, PZ, F8, FP1, F10, CP5",0.9015,0.7875,6.7875,0.3867,0.5095,0.2620,0.3757
6,7,FC1,"T5, PZ, F8, FP1, F10, CP5, FC1",0.8570,0.8375,8.0625,0.4474,0.5005,0.2529,0.4184
7,8,F9,"T5, PZ, F8, FP1, F10, CP5, FC1, F9",0.7549,0.7750,7.8750,0.4476,0.5218,0.2527,0.4120
8,9,CP1,"T5, PZ, F8, FP1, F10, CP5, FC1, F9, CP1",0.8473,0.7875,7.3875,0.3917,0.5448,0.2418,0.4065
9,10,FC2,"T5, PZ, F8, FP1, F10, CP5, FC1, F9, CP1, FC2",0.8920,0.8125,7.3875,0.4140,0.5462,0.2421,0.4153


Selected channels: ['T5', 'PZ', 'F8', 'FP1', 'F10', 'CP5', 'FC1', 'F9', 'CP1', 'FC2', 'P3', 'O1']


## 5. Fit Gen1 on every training seizure and test on every held-out-patient seizure

The four-patient pool is the complete Gen1 development set. The inferred fifth patient is the complete test set. No seizure from the test patient is used to fit the model or tune its alarm threshold.

In [6]:
gen1_config = replace(
    forecast_config,
    included_eeg_channels=tuple(selected_channels),
)
gen1_manifest = cohort_manifest.copy()
gen1_manifest["dataset_split"] = np.where(
    gen1_manifest["patient_id"].astype(str).eq(test_patient), "test", "development"
)
channel_hash = hashlib.sha1(
    ",".join(selected_channels).encode("utf-8")
).hexdigest()[:10]
training_tag = "-".join(patient.replace("PN", "") for patient in training_patients)
cache_tag = f"train{training_tag}_test{test_patient[2:]}_k{K}_{channel_hash}"
cache_dir = paths["processed"] / "k_suiter_leave_one_patient_out"

gen1_landmarks = rsf.build_landmark_dataset(
    gen1_manifest,
    cache_csv=cache_dir / f"{cache_tag}_landmarks.csv",
    cache_metadata_json=cache_dir / f"{cache_tag}_landmarks.json",
    config=gen1_config,
    force=FORCE_REBUILD_GEN1_FEATURES,
    verbose=True,
)
development = gen1_landmarks.loc[
    gen1_landmarks["patient_id"].astype(str).isin(training_patients)
].copy()
test = gen1_landmarks.loc[
    gen1_landmarks["patient_id"].astype(str).eq(test_patient)
].copy()
if development.empty or test.empty:
    raise RuntimeError("The patient-disjoint Gen1 split produced an empty partition.")
if set(development["patient_id"].astype(str).unique()) != set(training_patients):
    raise RuntimeError("Not every requested training patient reached Gen1.")
if set(test["patient_id"].astype(str).unique()) != {test_patient}:
    raise RuntimeError("The Gen1 test set is not patient-disjoint.")

gen1_model = rsf.fit_hazard_model(development, gen1_config)
predictions, metrics, patient_metrics, threshold, threshold_curve = rsf.evaluate_forecasts(
    gen1_model,
    development,
    test,
    warning_time_target=gen1_config.warning_time_target,
    config=gen1_config,
)

gen1_split_summary = pd.DataFrame([
    {
        "split": "development",
        "patients": ", ".join(training_patients),
        "seizures": development.loc[development["episode_type"].eq("preictal"), "source_event_id"].nunique(),
        "landmark_rows": len(development),
    },
    {
        "split": "test",
        "patients": test_patient,
        "seizures": test.loc[test["episode_type"].eq("preictal"), "source_event_id"].nunique(),
        "landmark_rows": len(test),
    },
])
display(gen1_split_summary)
display(metrics.style.format({"value": "{:.4f}"}))
display(patient_metrics.style.format({
    "sensitivity": "{:.4f}",
    "false_alarms_per_hour": "{:.4f}",
    "time_in_warning": "{:.4f}",
}))

Loaded 8,400 cached landmark rows.


,split,patients,seizures,landmark_rows
0,development,"PN00, PN06, PN10, PN12",24,7200
1,test,PN14,4,1200


,metric,value,interpretation
0,negative log likelihood,1.3658,Lower is better; proper score for the observed bin/no-event class.
1,multicategory Brier score,0.3716,Lower is better; probability error across 60 bins plus no-event.
2,multicategory Brier skill score,-0.0341,Above 0 improves on the development-set class-frequency forecast.
3,integrated survival Brier score,0.0921,Lower is better; mean survival-probability error across 5-minute horizon.
4,5-minute AUROC,0.3958,Discrimination only; does not assess calibration.
5,5-minute average precision,0.1672,Ranking metric sensitive to event prevalence.
6,conditional timing MAE (seconds),75.0000,Error of expected onset time among seizure landmarks.
7,seizure sensitivity,0.0000,Fraction of seizure episodes with an alarm after 18 consecutive high-risk landmarks.
8,time in warning,0.0000,Fraction of evaluated 5-second landmarks under warning.
9,false alarms per hour,0.0000,Rising persistent-alarm edges in interictal episodes per monitored hour.


,patient_id,landmarks,seizure_episodes,interictal_episodes,auroc,sensitivity,time_in_warning,false_alarms_per_hour,captured_seizures,total_seizures,median_warning_lead_seconds
0,PN14,1200,4,16,0.395812,0.0000,0.0000,0.0000,0.000000,4.000000,nan


## 6. Save the ranking and Gen1 accuracy outputs

In [ ]:
output_dir = paths["project"] / "results" / "k_suiter_leave_one_patient_out"
output_dir.mkdir(parents=True, exist_ok=True)
artifact_stem = f"train_{training_tag}_test_{test_patient[2:]}_k{K}"

artifact_paths = {
    "ranking_csv": output_dir / f"{artifact_stem}_ranking.csv",
    "metrics_csv": output_dir / f"{artifact_stem}_gen1_metrics.csv",
    "patient_metrics_csv": output_dir / f"{artifact_stem}_gen1_patient_metrics.csv",
    "predictions_csv": output_dir / f"{artifact_stem}_gen1_predictions.csv",
    "threshold_curve_csv": output_dir / f"{artifact_stem}_threshold_curve.csv",
    "summary_json": output_dir / f"{artifact_stem}_summary.json",
}
ranking.to_csv(artifact_paths["ranking_csv"], index=False)
metrics.to_csv(artifact_paths["metrics_csv"], index=False)
patient_metrics.to_csv(artifact_paths["patient_metrics_csv"], index=False)
predictions.to_csv(artifact_paths["predictions_csv"], index=False)
threshold_curve.to_csv(artifact_paths["threshold_curve_csv"], index=False)

summary = {
    "eligible_patients": list(ELIGIBLE_PATIENTS),
    "training_patients": list(training_patients),
    "test_patient": test_patient,
    "split_method": "leave-one-patient-out; no random seizure split",
    "training_seizures": int(gen1_split_summary.iloc[0]["seizures"]),
    "test_seizures": int(gen1_split_summary.iloc[1]["seizures"]),
    "k": K,
    "included_eeg_channels": selected_channels,
    "warning_threshold": float(threshold),
    "gen1_metrics": {
        str(row.metric): float(row.value) for row in metrics.itertuples()
    },
    "gen1_patient_metrics": patient_metrics.to_dict(orient="records"),
    "k_finder_result_path": str(k_finder_result.source_path),
    "artifacts": {name: path.name for name, path in artifact_paths.items() if name != "summary_json"},
}
artifact_paths["summary_json"].write_text(
    json.dumps(summary, indent=2) + "\n", encoding="utf-8"
)

print("Saved patient-disjoint experiment outputs:")
for path in artifact_paths.values():
    print(f"- {path}")

## Interpretation

- The sensor ranking uses only the four named training patients.
- The selected sensors must exist in all five eligible patients so the montage is deployable on the held-out patient; the held-out signals and labels are not used for ranking.
- Gen1 trains and tunes its warning policy on all seizures from the four training patients.
- Every seizure from the automatically inferred fifth patient is used for the final accuracy report.
- Change `TRAIN_PATIENTS` to run another one of the five possible leave-one-patient-out experiments.